## Prerequisites - GPU environment check

**Run the cell below first.** It checks the NVIDIA tools (`nvidia-smi`, `nvcc`, `nsys`, `ncu`) and the Python GPU packages (`numpy`, `numba`, `cupy`) this course needs, and prints how to fix anything missing.

- **OK** - ready to use.
- **MISSING** - a required tool; fix it before running this module.
- **optional** - only affects specific bonus paths; the workbooks skip these gracefully.

In [ ]:
# Prerequisites: verify the GPU toolchain before running this notebook.
# This finds check_gpu.py at the repository root and runs it.
import pathlib, runpy
_here = pathlib.Path.cwd()
for _d in (_here, *_here.parents):
    if (_d / "check_gpu.py").exists():
        runpy.run_path(str(_d / "check_gpu.py"), run_name="__main__")
        break
else:
    print("check_gpu.py not found - run `python check_gpu.py` from the repo root.")

# GPU-Accelerated Geant4: Celeritas and AdePT with TileCal Geometry

> Checked against upstream documentation in May 2026.
>
> Official Celeritas docs: https://celeritas-project.github.io/celeritas/user/index.html
> Celeritas quick start: https://celeritas-project.github.io/celeritas/user/index.html#quick-start-guide
> Geant4 integration examples: https://github.com/celeritas-project/celeritas/tree/main/example/geant4
> AdePT project: https://github.com/apt-sim/AdePT
> DD4hep project page: https://dd4hep.web.cern.ch/dd4hep/
> Geant4 documentation: https://geant4.web.cern.ch/documentation
> TileCal geometry and macro source: https://github.com/celeritas-project/atlas-tilecal-integration
>
> Note: the TileCal repository is still useful for the GDML geometry and macro cards, but its build and integration instructions predate the current Celeritas Geant4 API. In this notebook we use that repository only as a source of geometry and macro files.

Detector simulation is one of the largest consumers of CPU in HEP. The bottleneck is **electromagnetic transport** — the showers of e⁻/e⁺/γ inside calorimeters. Two CERN R&D projects offload exactly that hot path to the **GPU** while keeping the rest of the event on the CPU under Geant4:

| Engine | What it is | GPU EM transport via | License |
| --- | --- | --- | --- |
| **[Celeritas](https://github.com/celeritas-project/celeritas)** | GPU transport library with a Geant4 tracking-manager offload | its own physics + geometry (ORANGE / VecGeom) | Apache-2.0 |
| **[AdePT](https://github.com/apt-sim/AdePT)** | Lightweight Geant4 plugin that offloads EM transport | G4HepEm + VecGeom | Apache-2.0 |

Both are sibling efforts toward the same goal: run HEP simulation faster on heterogeneous hardware. This notebook has **two goals**:

1. **Run simulation on the GPU** — build and run the *same* physics on **CPU vs GPU** with both engines, and measure the speed-up.
2. **See where you can contribute** — these projects are active and welcome newcomers (final section).

Workflow:

1. Set up a prebuilt Geant4 + Celeritas toolchain from CVMFS (Key4hep) — no build.
2. Download the TileCal GDML geometry and macro files.
3. Build a small Geant4 application using the current Celeritas tracking-manager integration path.
4. **Celeritas:** compare CPU-only and GPU-enabled runs and inspect the diagnostics.
5. **AdePT:** run the standalone Geant4 + AdePT `example1` (GPU EM transport) from the CVMFS `devAdePT` view.
6. **Contribute:** the GPU-simulation landscape and good first issues.


## 0 Environment setup — Key4hep from CVMFS

This notebook targets a **Linux node with an NVIDIA GPU** and a mounted **CVMFS**. Instead of building Geant4/Celeritas from source, it uses the prebuilt **Key4hep** software stack from CVMFS, which ships Geant4, DD4hep, and Celeritas ready to use — so there is **no `spack install` and no long compile**.

The whole setup is a single `source`:

```bash
source /cvmfs/sw.hsf.org/key4hep/setup.sh
```

The next cell sources Key4hep and verifies the toolchain (Geant4, DD4hep, Celeritas, CMake, and the GPU).

**Recommended:** source Key4hep in your shell *before* launching Jupyter / VS Code so the Python kernel inherits the environment. The notebook does not rely on that, though — every `%%bash` build/run cell re-sources Key4hep, and the Python run helper does too, so you can run it top to bottom either way.

> Tip: `source /cvmfs/sw.hsf.org/key4hep/setup.sh -r <YYYY-MM-DD>` pins a specific release for reproducibility; the plain command uses the latest optimized build.


In [ ]:
%%bash
# Source the Key4hep software stack from CVMFS: prebuilt Geant4, DD4hep, and
# Celeritas -- no local build required. (Add "-r <YYYY-MM-DD>" to pin a release.)
source /cvmfs/sw.hsf.org/key4hep/setup.sh
set -e

echo "== Toolchain from Key4hep (CVMFS) =="
geant4-config --version && echo "geant4-config: OK"
command -v ddsim  >/dev/null 2>&1 && echo "DD4hep (ddsim): $(command -v ddsim)" || echo "DD4hep (ddsim): not on PATH"
if command -v celer-sim >/dev/null 2>&1 || command -v celer-g4 >/dev/null 2>&1; then
  echo "Celeritas: $(command -v celer-sim 2>/dev/null || command -v celer-g4)"
else
  echo "Celeritas: no celer-* tool on PATH (CMake find_package may still resolve it via CMAKE_PREFIX_PATH)"
fi
cmake --version | head -n1
command -v nvidia-smi >/dev/null 2>&1 \
  && nvidia-smi --query-gpu=name,driver_version --format=csv,noheader \
  || echo "nvidia-smi: not found (GPU runs will fall back to CPU)"


## 1 What the Key4hep stack provides

Sourcing `/cvmfs/sw.hsf.org/key4hep/setup.sh` puts a consistent, prebuilt HEP toolchain on your `PATH` and `CMAKE_PREFIX_PATH`:

- **Geant4** (11.x) — with the tracking-manager offload interface Celeritas needs.
- **DD4hep** — detector description (the minimal example here reads the GDML directly, but DD4hep matches the broader workflow).
- **Celeritas** — the GPU transport library, built with CUDA support in the CVMFS release.
- **CMake** and a C++17 compiler.

Because the stack is prebuilt on CVMFS, there is **no `spack install` and no long compile** — the setup cell above just sources it. If the verification cell reports that Celeritas is not present in the release you sourced, pin a newer one with `-r <YYYY-MM-DD>`.

GPU note: whether a run uses the GPU is decided at **runtime**, not build time. The comparison cells below toggle `CELER_DISABLE_DEVICE` to force CPU-only vs GPU execution of the *same* binary; if no NVIDIA GPU is visible, Celeritas falls back to CPU.

Each `%%bash` cell runs in a fresh shell, so it re-sources Key4hep before building or running, and the Python run helper wraps the executable in a Key4hep-sourced shell for the same reason. This means the notebook works whether or not you launched Jupyter from a Key4hep-sourced shell.


## 2 Download the TileCal geometry and macro files

The upstream TileCal repository still provides useful input files. The macro files we use here are `TBrun.mac` and `TBrun_all.mac`.

## 3 Write a minimal Geant4 plus Celeritas application

The current upstream integration pattern is:

1. Include `CeleritasG4.hh`.
2. Use `TrackingManagerIntegration::Instance()`.
3. Register `TrackingManagerConstructor` on your Geant4 physics list.
4. Call `BeginOfRunAction` and `EndOfRunAction` from a Geant4 run action.
5. Link with `Celeritas::G4` and use `celeritas_target_link_libraries(...)` in CMake.

The code below downloads the TileCal inputs, writes a small Geant4 application, and creates a matching `CMakeLists.txt`. The executable loads the GDML directly through Geant4's GDML parser and executes a Geant4 macro card.

In [ ]:
import pathlib
import urllib.request


tile_repo = "https://raw.githubusercontent.com/celeritas-project/atlas-tilecal-integration/main/"
for filename in (
    "TileTB_2B1EB_nobeamline.gdml",
    "TBrun.mac",
    "TBrun_all.mac",
):
    path = pathlib.Path(filename)
    if not path.exists():
        urllib.request.urlretrieve(tile_repo + filename, path)

cpp_source = r'''#include <memory>
#include <string>

#include <FTFP_BERT.hh>
#include <G4GDMLParser.hh>
#include <G4ParticleGun.hh>
#include <G4ParticleTable.hh>
#include <G4RunManagerFactory.hh>
#include <G4SystemOfUnits.hh>
#include <G4ThreeVector.hh>
#include <G4UImanager.hh>
#include <G4UserRunAction.hh>
#include <G4VUserActionInitialization.hh>
#include <G4VUserDetectorConstruction.hh>
#include <G4VUserPrimaryGeneratorAction.hh>

#include <CeleritasG4.hh>

using TMI = celeritas::TrackingManagerIntegration;

namespace
{
class DetectorConstruction final : public G4VUserDetectorConstruction
{
  public:
    G4VPhysicalVolume* Construct() final
    {
        parser_.Read("TileTB_2B1EB_nobeamline.gdml", false);
        return parser_.GetWorldVolume();
    }

  private:
    G4GDMLParser parser_;
};

class PrimaryGeneratorAction final : public G4VUserPrimaryGeneratorAction
{
  public:
    PrimaryGeneratorAction()
        : gun_(1)
    {
        auto* particle
            = G4ParticleTable::GetParticleTable()->FindParticle("e-");
        gun_.SetParticleDefinition(particle);
        gun_.SetParticleEnergy(18 * GeV);
        gun_.SetParticlePosition(G4ThreeVector{0, 0, 0});
        gun_.SetParticleMomentumDirection(G4ThreeVector{1, 0, 0});
    }

    void GeneratePrimaries(G4Event* event) final
    {
        gun_.GeneratePrimaryVertex(event);
    }

  private:
    G4ParticleGun gun_;
};

class RunAction final : public G4UserRunAction
{
  public:
    void BeginOfRunAction(G4Run const* run) final
    {
        TMI::Instance().BeginOfRunAction(run);
    }

    void EndOfRunAction(G4Run const* run) final
    {
        TMI::Instance().EndOfRunAction(run);
    }
};

class ActionInitialization final : public G4VUserActionInitialization
{
  public:
    void BuildForMaster() const final
    {
        this->SetUserAction(new RunAction{});
    }

    void Build() const final
    {
        this->SetUserAction(new PrimaryGeneratorAction{});
        this->SetUserAction(new RunAction{});
    }
};

celeritas::SetupOptions MakeOptions()
{
    celeritas::SetupOptions options;
    options.max_num_tracks = 4096;
    options.initializer_capacity = 4096 * 128;
    options.ignore_processes = {"CoulombScat"};
    options.output_file = "tile_gpu.out.json";
    return options;
}
}  // namespace

int main(int argc, char* argv[])
{
    std::string macro = argc > 1 ? argv[1] : "TBrun.mac";

    std::unique_ptr<G4RunManager> run_manager{
        G4RunManagerFactory::CreateRunManager(G4RunManagerType::Default)};

    run_manager->SetUserInitialization(new DetectorConstruction{});

    auto& tmi = TMI::Instance();
    auto* physics_list = new FTFP_BERT{/* verbosity = */ 0};
    physics_list->RegisterPhysics(
        new celeritas::TrackingManagerConstructor(&tmi));
    run_manager->SetUserInitialization(physics_list);
    run_manager->SetUserInitialization(new ActionInitialization{});

    tmi.SetOptions(MakeOptions());

    run_manager->Initialize();
    G4UImanager::GetUIpointer()->ApplyCommand("/control/execute " + macro);

    return 0;
}
'''

cmake_source = r'''cmake_minimum_required(VERSION 3.18...4.1)
project(tile_gpu LANGUAGES CXX)

find_package(Celeritas 0.6 REQUIRED)
find_package(Geant4 REQUIRED)

if(NOT CELERITAS_USE_Geant4)
  message(FATAL_ERROR "This Celeritas installation was not built with Geant4 support")
endif()

add_executable(tile_gpu tile_gpu.cc)
target_compile_features(tile_gpu PRIVATE cxx_std_17)

celeritas_target_link_libraries(tile_gpu
  Celeritas::G4
  ${Geant4_LIBRARIES}
)
'''

pathlib.Path("tile_gpu.cc").write_text(cpp_source)
pathlib.Path("CMakeLists.txt").write_text(cmake_source)

print("Prepared input files:")
for path in [
    pathlib.Path("TileTB_2B1EB_nobeamline.gdml"),
    pathlib.Path("TBrun.mac"),
    pathlib.Path("TBrun_all.mac"),
    pathlib.Path("tile_gpu.cc"),
    pathlib.Path("CMakeLists.txt"),
]:
    print(" -", path)
print("CMake is configured for Celeritas 0.6 or newer.")

## 4 Configure and build

This replaces the older hand-written `g++` command that relied on nonstandard environment variables such as `CELERITAS_INCLUDE_DIRS`. The current upstream recommendation is to use CMake plus `find_package(Celeritas ...)`.

Using `0.6` here keeps the example compatible with current stable Celeritas installs while still matching the tracking-manager integration API used in this notebook.

If your Celeritas build includes MPI support and you only want a single-process notebook run, set `CELER_DISABLE_PARALLEL=1` before executing the benchmark cells.

In [ ]:
%%bash
# Source Key4hep first (it may reference unset vars), then enable strict mode.
source /cvmfs/sw.hsf.org/key4hep/setup.sh
set -e

# Celeritas and Geant4 are found via CMAKE_PREFIX_PATH set by Key4hep.
cmake -S . -B build -DCMAKE_BUILD_TYPE=Release
cmake --build build -j


In [ ]:
import os
import subprocess
import time

KEY4HEP_SETUP = "/cvmfs/sw.hsf.org/key4hep/setup.sh"


def run_tile(macro: str, use_gpu: bool):
    env = os.environ.copy()
    env.setdefault("CELER_DISABLE_PARALLEL", "1")
    if not use_gpu:
        env["CELER_DISABLE_DEVICE"] = "1"
    else:
        env.pop("CELER_DISABLE_DEVICE", None)

    # Run the executable inside a shell that sources Key4hep, so the Geant4 and
    # Celeritas runtime libraries (from CVMFS) are on the library path -- no
    # matter how Jupyter itself was launched. CELER_DISABLE_DEVICE is passed
    # through the environment and honoured at runtime.
    cmd = f'source "{KEY4HEP_SETUP}" >/dev/null 2>&1; exec ./build/tile_gpu "{macro}"'
    t0 = time.perf_counter()
    result = subprocess.run(
        ["bash", "-c", cmd],
        env=env,
        text=True,
        capture_output=True,
        check=True,
    )
    dt = time.perf_counter() - t0
    return dt, result.stdout, result.stderr


cpu_t, cpu_out, cpu_err = run_tile("TBrun.mac", use_gpu=False)
gpu_t, gpu_out, gpu_err = run_tile("TBrun.mac", use_gpu=True)

print(f"CPU wall time: {cpu_t:.2f} s")
print(f"GPU wall time: {gpu_t:.2f} s")
if gpu_t > 0:
    print(f"Speed-up: {cpu_t / gpu_t:.2f}x")


## 5 Inspect the Celeritas diagnostics output

The example above writes `tile_gpu.out.json`. Its exact contents depend on the Celeritas version and enabled diagnostics, so the safest first step is to inspect the top-level structure rather than hard-code field names.

In [ ]:
import json
import pathlib
from pprint import pprint

path = pathlib.Path("tile_gpu.out.json")
if not path.exists():
    raise FileNotFoundError("Run the executable first so tile_gpu.out.json exists")

with path.open() as handle:
    diagnostics = json.load(handle)

print("Top-level keys:")
for key in sorted(diagnostics):
    print(" -", key)

print("\nSelected preview:")
preview = {key: diagnostics[key] for key in list(diagnostics)[:3]}
pprint(preview)

## 6 Exercises

### Exercise 1: Mixed-particle macro

Run `TBrun_all.mac`, which contains electron, pion, kaon, and proton runs from the upstream TileCal repository.

Questions:

1. Does the overall GPU speed-up decrease compared with `TBrun.mac`?
2. Why is that expected for a workload with a larger hadronic component?

In [ ]:
cpu_t, _, _ = run_tile("TBrun_all.mac", use_gpu=False)
gpu_t, _, _ = run_tile("TBrun_all.mac", use_gpu=True)
print(f"TBrun_all.mac speed-up: {cpu_t / gpu_t:.2f}x")

### Exercise 2: Event-count scaling

Make a copy of `TBrun.mac`, increase the event count, and see whether the GPU run benefits more from the larger workload.

Expected discussion point: GPUs usually amortize setup costs better as the problem size grows, but hadronic-heavy transport can still cap the gain.

Further reading:

- Celeritas user documentation: https://celeritas-project.github.io/celeritas/user/index.html
- Celeritas Geant4 examples: https://github.com/celeritas-project/celeritas/tree/main/example/geant4
- DD4hep project documentation: https://dd4hep.web.cern.ch/dd4hep/
- Geant4 documentation portal: https://geant4.web.cern.ch/documentation
- TileCal geometry repository used for this lesson: https://github.com/celeritas-project/atlas-tilecal-integration

In [ ]:
from pathlib import Path

source = Path("TBrun.mac").read_text()
Path("TBrun_100k.mac").write_text(source.replace("/run/beamOn 10000", "/run/beamOn 100000"))

cpu_t, _, _ = run_tile("TBrun_100k.mac", use_gpu=False)
gpu_t, _, _ = run_tile("TBrun_100k.mac", use_gpu=True)
print(f"TBrun_100k.mac speed-up: {cpu_t / gpu_t:.2f}x")

## 7 AdePT — a second GPU EM-transport engine

Celeritas above offloaded EM transport by *replacing* Geant4's transport with its own on the GPU. **[AdePT](https://github.com/apt-sim/AdePT)** takes a complementary route: it is a **Geant4 plugin** that keeps Geant4 in charge and offloads the e⁻/e⁺/γ EM shower to the GPU using **G4HepEm** physics on **VecGeom** geometry. Same goal (move the EM hot path to the GPU), different implementation — worth seeing both.

AdePT ships a standalone Geant4 application, `example1`, that we run straight from CVMFS. Source the LCG `devAdePT` view (separate from Key4hep):

```bash
source /cvmfs/sft.cern.ch/lcg/views/devAdePT/latest/x86_64-el9-gcc13-opt/setup.sh
```

The helper module `06/adept-demo/adept_demo.py` sources that view for you and provides:

- `verify_adept()` — report `example1`, Geant4, `nvcc`, and the GPU seen from the view.
- `run_example1(run=...)` — run the GPU EM-transport example (detection-only by default; opt in with `run=True`).

**CPU vs GPU with AdePT:** `example1` runs the EM shower on the GPU; a standard-Geant4 build of the same app (without the AdePT physics constructor) is the CPU baseline — the same CPU-vs-GPU contrast the Celeritas cells measured with `CELER_DISABLE_DEVICE`. On the school GPU node, run the cells below with `run=True`; off a GPU node they print the one-time build commands and skip cleanly.

> Requirements (all satisfied by the `devAdePT` view): Geant4 > 11, VecGeom ≥ 2.0.0-rc.4, G4HepEm, CUDA > 12, C++20. If `example1` is not prebuilt in the view, the helper prints the one-time `git clone` + `cmake` build commands.


In [ ]:
import sys
import pathlib

# Make the factored AdePT helper importable (like 05/gen-demo and 07/gpu-demo).
sys.path.insert(0, str(pathlib.Path("adept-demo").resolve()))
import adept_demo as ad

# Report the AdePT toolchain visible from the devAdePT CVMFS view.
ad.verify_adept()


In [ ]:
# Detection-only: locate the standalone Geant4 + AdePT `example1`.
# On the school GPU node (with the devAdePT view), re-run with run=True to launch
# the GPU EM-transport example and time it:
#     ad.run_example1(macro="example1.mac", run=True)
ad.run_example1(run=False)


## 8 Where you can contribute

GPU-accelerated detector simulation is an **active, open R&D area** — a great place for newcomers to make a real contribution. All four projects below are open source and take external pull requests.

| Project | Role in the GPU stack | Good first contributions | Links |
| --- | --- | --- | --- |
| **Celeritas** | GPU transport library + Geant4 offload | reproduce/report benchmarks, add validation plots, docs, physics tests | [repo](https://github.com/celeritas-project/celeritas) · [issues](https://github.com/celeritas-project/celeritas/issues) |
| **AdePT** | Geant4 plugin offloading EM transport | run `example1` on new geometries, profile kernels, add examples, docs | [repo](https://github.com/apt-sim/AdePT) · [issues](https://github.com/apt-sim/AdePT/issues) |
| **G4HepEm** | Compact EM physics used by AdePT | add/verify physics processes, unit tests, table validation | [repo](https://github.com/mnovak42/g4hepem) |
| **VecGeom** | Vectorized/GPU geometry used by both | geometry unit tests, shape implementations, benchmarks | [CERN GitLab](https://gitlab.cern.ch/VecGeom/VecGeom) |

How to get started:

1. **Reproduce a result.** Run the Celeritas and AdePT examples in this notebook on a GPU node and record the CPU-vs-GPU speed-up for your detector / particle mix. Reproducibility reports are genuinely valued.
2. **Pick a "good first issue".** Both Celeritas and AdePT tag beginner-friendly issues on GitHub.
3. **Bring HEP physics.** Try your own GDML geometry or a new particle / energy, and report where GPU transport helps most (EM-dominated showers) and least (hadronic-heavy events) — exactly the trade-off Exercises 1–2 probed.
4. **Improve docs and examples.** New, working, well-documented examples are one of the most useful contributions to a fast-moving project.

Ideas specific to this course:

- Extend `06/adept-demo/` with a CPU-baseline build of `example1` (standard Geant4, no AdePT physics) so the AdePT CPU-vs-GPU number is measured end to end, as the Celeritas cells do.
- Add a scan over particle type / energy and plot speed-up vs the EM fraction of the shower.


## 9 Clean-up


In [ ]:
%%bash
rm -rf build
rm -f tile_gpu.cc CMakeLists.txt tile_gpu.out.json TBrun.mac TBrun_all.mac TBrun_100k.mac